# 50 - chaos, ramp and soak on the T4 (Build Guide Phase 14, steps 1-3)
TuringServe at tag `m3-rc`, Qwen2.5-1.5B, one T4. Order: install the pinned stack, start the server with `--debug-endpoints`, chaos test, ramp (maximum rate with >= 90% goodput), soak at 70% of that rate.

**Best run as a batch:** set Accelerator to GPU T4 x2 and Internet on, then **Save Version -> Save & Run All (Commit)**. A batch run keeps going when the browser tab is closed or the laptop sleeps; the files appear under the saved version's Output tab when it finishes. It also works interactively with Run All if you keep the tab open.

Time: about 25 minutes of setup, chaos and ramp, plus the soak (`SOAK_HOURS` in the first cell; 1.0 means about 1 h 40 min in total, 2.0 about 2 h 40 min). Do not use *Factory reset*.

In [ ]:
import os, sys, json, time, signal, shutil, subprocess, threading, glob

# ---- the only settings you may want to change ----
SOAK_HOURS = 1.0         # the Build Guide asks for 2.0; 1.0 still gives a meaningful memory-growth trend (growth is measured after minute 10)
RUN_CHAOS = True
RUN_RAMP = True
# --------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
BIG = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"     # large files stay out of the saved output
MODEL = f"{BIG}/models/Qwen2.5-1.5B-Instruct"
SHAREGPT = f"{BIG}/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
REPO = f"{WORK}/turingserve"
RESULTS = f"{WORK}/results"
PORT = 8000
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_SHAREGPT"] = SHAREGPT
os.environ["TS_KAGGLE"] = "1"
srv = None
T_START = time.time()


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "turingserve serve"], capture_output=True)
    time.sleep(3)


def stream(cmd, log_name, heartbeat=None, cwd=REPO):
    """Run a command, echo its output live, keep a copy in results/<log_name>, return the exit code."""
    t0 = time.time()
    done = threading.Event()
    if heartbeat:
        def beat():
            while not done.wait(heartbeat):
                print(f"  ... still running ({(time.time() - t0) / 60:.0f} min)", flush=True)
        threading.Thread(target=beat, daemon=True).start()
    p = subprocess.Popen(cmd, cwd=cwd, env=dict(os.environ, PYTHONUNBUFFERED="1"),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(f"{RESULTS}/{log_name}", "w") as f:
        for line in p.stdout:
            if "Token indices sequence length" in line or "Running this sequence" in line:
                continue
            print(line, end="", flush=True)
            f.write(line)
    p.wait()
    done.set()
    print(f"[{log_name}] exit code {p.returncode}, {(time.time() - t0) / 60:.1f} min\n")
    return p.returncode


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB;", f"soak hours: {SOAK_HOURS}")

## 1. Pin the stack (torch 2.6.0+cu124, Triton 3.2.0)

In [ ]:
def torch_version():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
    return r.stdout.strip()

print("torch before:", torch_version())
if torch_version() != "2.6.0+cu124":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], capture_output=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url",
                        "https://download.pytorch.org/whl/cu124"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError("torch pin failed")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45,<5", "safetensors", "fastapi",
                    "uvicorn[standard]", "httpx", "openai", "prometheus-client", "psutil", "datasets", "pandas", "matplotlib"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise RuntimeError("dependency install failed")
print("torch after:", torch_version())

## 2. Model, dataset and repository at tag `m3-rc`

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

if not os.path.exists(f"{MODEL}/model.safetensors"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL)
print("model ok:", os.path.exists(f"{MODEL}/model.safetensors"))
if RUN_RAMP and not os.path.exists(SHAREGPT):
    hf_hub_download("anon8231489123/ShareGPT_Vicuna_unfiltered", "ShareGPT_V3_unfiltered_cleaned_split.json",
                    repo_type="dataset", local_dir=f"{BIG}/datasets/sharegpt")
print("dataset ok:", os.path.exists(SHAREGPT))
if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO, capture_output=True)
r = subprocess.run([sys.executable, "scripts/check_versions.py"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-800:])
open(f"{RESULTS}/versions.txt", "w").write(r.stdout)

## 3. Start TuringServe with the debug endpoints
The first start on a new machine also runs the kernel tuning (several minutes).

In [ ]:
import requests
stop_server()
log_path = f"{RESULTS}/server.log"
srv = subprocess.Popen(["turingserve", "serve", "--model", MODEL, "--served-model-name", "qwen", "--port", str(PORT),
                        "--debug-endpoints"], stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
for i in range(240):
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
            print(f"server up after {time.time() - t0:.0f}s, pid {srv.pid}")
            break
    except Exception:
        pass
    if srv.poll() is not None:
        print(open(log_path).read()[-3000:])
        raise RuntimeError("server exited during startup, log tail above")
    if i % 12 == 11:
        print(f"  still starting... {time.time() - t0:.0f}s")
    time.sleep(5)
else:
    print(open(log_path).read()[-3000:])
    raise RuntimeError("server did not become healthy in 20 minutes")
print("debug state:", requests.get(f"http://localhost:{PORT}/debug/state", timeout=5).json())

## 4. Chaos test
Pass: no crash, every error is a 4xx from ARCH 6.1, `/debug/state` clean afterwards.

In [ ]:
chaos_rc = None
if RUN_CHAOS:
    chaos_rc = stream([sys.executable, "-m", "benchmarks.chaos", "--base-url", f"http://localhost:{PORT}", "--model", "qwen"],
                      "chaos.txt", heartbeat=120)
    import requests
    print("server still alive after chaos:", srv.poll() is None)
    print("debug state after chaos:", requests.get(f"http://localhost:{PORT}/debug/state", timeout=10).json())
else:
    print("chaos skipped")

## 5. Ramp (0.5 to 8 req/s, 60 s each)
The soak rate is 70% of the highest rate that still has >= 90% goodput and no errors.

In [ ]:
soak_rate = 1.0
if RUN_RAMP:
    rc = stream([sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}", "--engine", "turingserve",
                 "--model", "qwen", "--tokenizer", MODEL, "--dataset", "sharegpt", "--num-prompts", "300", "--mode", "ramp",
                 "--ramp", "0.5,1,2,3,4,6,8", "--step-seconds", "60", "--repeat", "1", "--warmup", "5", "--ignore-eos",
                 "--gpu-log", "--out", RESULTS, "--label", "t4-1.5b-ramp"], "ramp.txt", heartbeat=120)
    files = sorted(glob.glob(f"{RESULTS}/*_turingserve_sharegpt_t4-1.5b-ramp.json"))
    if rc == 0 and files:
        d = json.load(open(files[-1]))
        print(f"{'rate req/s':>11}{'goodput':>10}{'errors':>8}{'tok/s':>9}")
        ok_rates = []
        for r in d["runs"]:
            m = r["median"]
            print(f"{r['level']:>11}{m['goodput_fraction']:>10.0%}{m['errors']:>8}{m['output_throughput_tok_s']:>9.1f}")
            if m["goodput_fraction"] >= 0.9 and m["errors"] == 0:
                ok_rates.append(float(r["level"]))
        max_rate = max(ok_rates) if ok_rates else 0.5
        soak_rate = max(0.3, round(0.7 * max_rate, 2))
        print(f"maximum rate with >= 90% goodput: {max_rate} req/s -> soak rate (70%): {soak_rate} req/s")
    else:
        print("ramp failed; using the default soak rate", soak_rate)
else:
    print("ramp skipped; soak rate", soak_rate)

## 6. Soak
Pass: RSS growth < 5% after the first 10 minutes, `num_requests_tracked == 0` at the end, all blocks free or cached, zero 5xx.

In [ ]:
print(f"soak: {soak_rate} req/s for {SOAK_HOURS} h; started at {time.strftime('%H:%M:%S')} (prints a line every 5 minutes)")
soak_rc = stream([sys.executable, "-m", "benchmarks.soak", "--base-url", f"http://localhost:{PORT}", "--model", "qwen",
                  "--server-pid", str(srv.pid), "--rate", str(soak_rate), "--hours", str(SOAK_HOURS),
                  "--out", f"{RESULTS}/soak.json"], "soak.txt", heartbeat=300)

## 7. Summary and download
Download **`/kaggle/working/soak_chaos_bundle.zip`** (or take it from the saved version's Output tab).

In [ ]:
stop_server()
print("chaos:", {None: "skipped", 0: "PASS (exit 0)"}.get(chaos_rc, f"FAIL (exit {chaos_rc})"))
print("soak :", "PASS" if soak_rc == 0 else f"FAIL (exit {soak_rc})")
try:
    s = json.load(open(f"{RESULTS}/soak.json"))
    rss = [x["rss"] for x in s["samples"]]
    print("soak requests:", s["stats"])
    print(f"RSS growth after minute 10: {s['rss_growth']:+.2%}   (start {rss[0] / 1e6:.0f} MB, end {rss[-1] / 1e6:.0f} MB, max {max(rss) / 1e6:.0f} MB)")
    print("final server state:", s["final"])
except Exception as e:
    print("no soak.json:", e)

bundle = f"{WORK}/soak_chaos_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*"):
    if os.path.isfile(f):
        shutil.copy(f, bundle)
shutil.make_archive(bundle, "zip", bundle)
print("\nDownload:", bundle + ".zip", f"({os.path.getsize(bundle + '.zip') / 1e6:.1f} MB)",
      f"- total notebook time {(time.time() - T_START) / 60:.0f} min")